# Import

In [ ]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.messages import convert_to_openai_messages, convert_to_messages
from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List
from IPython.display import Image, display
from operator import add
from openai import OpenAI
import random
import ast
import inspect
import instructor
import json
# from utils.utils import get_tool_descriptions, format_ai_message

# Single Node Graph

In [ ]:
class State(BaseModel):
    message: str
    answer: str = ""
    vibe: str

In [ ]:
def append_vibes_to_query(state: State) -> dict:
    return {
        "answer": f"{state.message} {state.vibe}"
    }

In [ ]:
workflow = StateGraph[State, None, State, State](State)
workflow.add_node("append_vibes_to_query", append_vibes_to_query)
workflow.add_edge(START, "append_vibes_to_query")
workflow.add_edge("append_vibes_to_query", END)
graph =workflow.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = {
    "message": "Give me some vibes!",
    "vibe": "I'm feeling like a badass today!"
}

In [ ]:
result = graph.invoke(initial_state)


In [ ]:
result

In [ ]:
initial_state = {
    "message": "ABC",
    "vibe": "I'm feeling like a badass today!"
}

In [ ]:
result = graph.invoke(initial_state)

In [ ]:
result

# Conditional Graph

In [ ]:
class State(BaseModel):
    message: str
    answer: str = ""

In [ ]:
def append_vibes_to_query(state: State) -> dict:
    return {
        "answer": "I am here to add some vibes:"
    }

In [ ]:
def router(state: State) -> Literal["append_vibe_1", "append_vibe_2", "append_vibe_3"]:
    vibes = ["append_vibe_1", "append_vibe_2", "append_vibe_3"]
    vibe_path = random.choice(vibes)
    return vibe_path

In [ ]:
def append_vibe_1(state: State) -> dict:
    vibe = "I'm feeling like a badass today!"
    return {"answer": f"{state.answer} {vibe}"}


def append_vibe_2(state: State) -> dict:
    vibe = "I'm feeling like a boss today!"
    return {"answer": f"{state.answer} {vibe}"}


def append_vibe_3(state: State) -> dict:
    vibe = "I'm feeling like a legend today!"
    return {"answer": f"{state.answer} {vibe}"}

In [ ]:
workflow = StateGraph[State, None, State, State](State)
workflow.add_node("append_vibes_to_query", append_vibes_to_query)
workflow.add_node("append_vibe_1", append_vibe_1)
workflow.add_node("append_vibe_2", append_vibe_2)
workflow.add_node("append_vibe_3", append_vibe_3)
workflow.add_conditional_edges(
    "append_vibes_to_query",
    router,
    {"append_vibe_1": "append_vibe_1", "append_vibe_2": "append_vibe_2", "append_vibe_3": "append_vibe_3"},
)
workflow.add_edge(START, "append_vibes_to_query")
workflow.add_edge("append_vibe_1", END)
workflow.add_edge("append_vibe_2", END)
workflow.add_edge("append_vibe_3", END)
graph = workflow.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = {
    "message": "I am here to add some vibes:",
}

In [ ]:
result = graph.invoke(initial_state)


In [ ]:
result

# Agent Graph